# Prueba técnica ingeniero de datos

## Librerias

In [1]:
import pandas as pd

## Funciones auxiliares

In [ ]:
def get_secrets_from_vault() -> dict[str, str]:
    """
    Recupera secretos desde Databricks, Fabric o Azure Key Vault (local).
    Detecta el entorno automáticamente.
    """
    key_vault = {
        "url": "https://kv-aeu-ecp-dev-energysys.vault.azure.net/",
        "db_scope": "sc-akv-dev-hubvfv",
    }
    vault_key_dict = {
        "db_token_databricks": "hub-db-dev-pid2-token",
        
        "spo_hub_secret": "hub-spo-equipohub-client-secret",
        "spo_hub_id": "hub-spo-equipohub-client-id",
        "spo_hub_url": "hub-spo-equipohub-site-url",

        "az_fabric_tenant" : "hub-az-powerBI-API-APP-tenant-id",
        "az_fabric_id" : "hub-az-powerBI-API-APP-client-id",
        "az_fabric_secret" :"hub-az-powerBI-API-APP-client-secret"
    }

    # Detectar entorno
    if "DATABRICKS_RUNTIME_VERSION" in os.environ:
        env = "databricks"
    else:
        try:
            import notebookutils  # noqa
            env = "fabric"
        except ImportError:
            env = "local"

    print("environment =", env)
    secret_dict = {}

    if env == "databricks":
        try:
            dbutils = globals()["dbutils"]
        except KeyError:
            from IPython import get_ipython
            dbutils = get_ipython().user_ns.get("dbutils")

        for key, val in vault_key_dict.items():
            secret_dict[key] = dbutils.secrets.get(
                scope=key_vault["db_scope"],
                key=val,
            )

    elif env == "fabric":
        from notebookutils import mssparkutils

        for key, val in vault_key_dict.items():
            secret_dict[key] = mssparkutils.credentials.getSecret(
                key_vault["url"],
                val,
            )

    else:  # local
        from azure.identity import DefaultAzureCredential
        from azure.keyvault.secrets import SecretClient

        credential = DefaultAzureCredential()
        client = SecretClient(vault_url=key_vault["url"], credential=credential)

        for key, val in vault_key_dict.items():
            secret_dict[key] = client.get_secret(val).value

    return secret_dict